### SEEING


In [15]:
import numpy as np
import matplotlib.pyplot as plt
import os

from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from scipy.ndimage import gaussian_filter
from scipy.optimize import curve_fit


calaremos el seeing, utilizando el FWHM. El perfil de brillo de una estrella cumple una funcion gaussiana

In [21]:
#abrimos los fits
star_1b = fits.getdata('HIP_6257_B_combinada.fits')
star_1v = fits.getdata('HIP_6257_V_combinada.fits')
star_2b = fits.getdata('HIP_115796_B_combinada.fits')
star_2v = fits.getdata('HIP_115796_V_combinada.fits')

#extraemos los headers para obtener la masa de arire
h_1b = fits.getheader('HIP_6257_B_combinada.fits')
h_1v = fits.getheader('HIP_6257_V_combinada.fits')
h_2b = fits.getheader('HIP_115796_B_combinada.fits')
h_2v = fits.getheader('HIP_115796_V_combinada.fits')

In [ ]:
def gauss2d(coords, A, xc, yc, sx, sy, th, B):
    x, y = coords
    c, s = np.cos(th), np.sin(th)
    u = (x-xc)*c + (y-yc)*s
    v = -(x-xc)*s + (y-yc)*c
    return (A*np.exp(-0.5*(u**2/sx**2 + v**2/sy**2)) + B).ravel()
 
'''ubicamos la estrella mas brillante y se ajusta a una gaussiana 2D elipsoidal (usando el con cielo como parametro)
     esto devuelve FWHM (px), elongacion (eje mayor/menor) y centro'''
def medir_fwhm(data, caja=40, fwhm_ini=10.0):
   
    # rellenamos los NaN reemplazandolos x la mediana de la imagen 
    lleno = np.where(np.isfinite(data), data, np.nanmedian(data))    
    #suavizamos la imagen (para evitar pixeles calientes) y buscamos el pixel mas brillante 
    suav = gaussian_filter(lleno, fwhm_ini/2.3548)                 
    yp, xp = np.unravel_index(np.argmax(suav), suav.shape)

    y0, x0 = max(yp-caja, 0), max(xp-caja, 0)
    rec = lleno[y0:yp+caja+1, x0:xp+caja+1]
    Y, X = np.mgrid[y0:y0+rec.shape[0], x0:x0+rec.shape[1]]

    #usado clipping, medimos el cielo inicial --> borde del recorte
    R = np.hypot(X-xp, Y-yp)
    _, cielo, _ = sigma_clipped_stats(rec[R > 0.7*caja], sigma=3.0)

    # buscamos la campana q mejor se ajuste a los px de la caja, con el ceilo como parametro
    s0 = fwhm_ini/2.3548
    p0 = [rec.max()-cielo, xp, yp, s0, s0, 0.0, cielo]
    lo = [0, xp-5, yp-5, 0.5, 0.5, -np.pi, -np.inf]
    hi = [np.inf, xp+5, yp+5, caja, caja, np.pi, np.inf]
    p, _ = curve_fit(gauss2d, (X, Y), rec.ravel(), p0=p0, bounds=(lo, hi))

    #calculamos el FWHM
    sx, sy = abs(p[3]), abs(p[4])
    return 2.3548*np.sqrt(sx*sy), max(sx, sy)/min(sx, sy), p[1], p[2]

In [ ]:
PIXSCALE = 0.317   #  solucion de astrometry.net (el 0.36 del header es nominal)

#aplicamos a cada estrella y multiplicamos por la escala del pixel -> pasndo a arcsec
estrellas = {'HIP 6257 B':   (star_1b, h_1b),
             'HIP 6257 V':   (star_1v, h_1v),
             'HIP 115796 B': (star_2b, h_2b),
             'HIP 115796 V': (star_2v, h_2v)}

resultados = {}
for nombre, (img, hdr) in estrellas.items():
    fwhm_px, elong, x, y = medir_fwhm(img)
    resultados[nombre] = (fwhm_px*PIXSCALE, elong, hdr.get('AIRMASS'))
    print(f"{nombre:13s} FWHM = {fwhm_px:5.2f} px = {fwhm_px*PIXSCALE:4.2f}\" | "
          f"elongacion = {elong:4.2f} | X = {hdr.get('AIRMASS')} | centro = ({x:.1f}, {y:.1f})")

HIP 6257 B    FWHM = 10.41 px = 3.30" | elongacion = 1.06 | X = 1.23234957968238 | centro = (2030.4, 2039.4)
HIP 6257 V    FWHM =  9.46 px = 3.00" | elongacion = 1.09 | X = 1.20200608134615 | centro = (2028.3, 2037.9)
HIP 115796 B  FWHM =  9.40 px = 2.98" | elongacion = 1.11 | X = 1.05339945495237 | centro = (2057.5, 2050.0)
HIP 115796 V  FWHM = 10.71 px = 3.39" | elongacion = 1.09 | X = 1.03860877333398 | centro = (2060.2, 2052.9)


In [24]:
for filtro in ('B', 'V'):
    vals = [v[0] for k, v in resultados.items() if k.endswith(filtro)]
    print(f'Seeing filtro {filtro}: media = {np.mean(vals):.2f}"  (rango {min(vals):.2f}" - {max(vals):.2f}")')

Seeing filtro B: media = 3.14"  (rango 2.98" - 3.30")
Seeing filtro V: media = 3.20"  (rango 3.00" - 3.39")
